<a href="https://colab.research.google.com/github/ast116/Medical_OCR-NLP_System/blob/main/NLP_spaCy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [3]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving medical_ocr&nlp_system.zip to medical_ocr&nlp_system.zip
User uploaded file "medical_ocr&nlp_system.zip" with length 46409390 bytes


In [4]:
import zipfile
import io

# Remplacez 'mon_projet.zip' par le nom de votre fichier compressé
zip_file_name = next(iter(uploaded)) # Prend le premier fichier téléchargé

with zipfile.ZipFile(io.BytesIO(uploaded[zip_file_name]), 'r') as zf:
    zf.extractall('/content/medical_ocr&nlp_system') # Crée un nouveau dossier pour le projet décompressé

print(f"Le projet a été décompressé dans '/content/medical_ocr&nlp_system'")

# Lister le contenu pour vérifier (optionnel)
import os
print(os.listdir('/content/medical_ocr&nlp_system'))

Le projet a été décompressé dans '/content/medical_ocr&nlp_system'
['medical_ocr&nlp_system']


In [5]:
from pathlib import Path
import os

# Chemin local après unzip
PROJECT_DIR = Path('/content/medical_ocr&nlp_system')

# Si le zip contient un sous-dossier racine, on le détecte automatiquement
if not (PROJECT_DIR / 'data').exists():
    subs = [p for p in PROJECT_DIR.iterdir() if p.is_dir()]
    for p in subs:
        if (p / 'data').exists():
            PROJECT_DIR = p
            break

DATA_DIR = PROJECT_DIR / 'data'
OCR_DIR = DATA_DIR / 'ocr_output'
NER_GOLD_DIR = DATA_DIR / 'ner_gold'

NER_GOLD_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR exists =', DATA_DIR.exists())
print('OCR_DIR exists =', OCR_DIR.exists())
print('NER_GOLD_DIR =', NER_GOLD_DIR)
print('OCR files count =', len(list(OCR_DIR.glob('*.txt'))) if OCR_DIR.exists() else 0)


PROJECT_DIR = /content/medical_ocr&nlp_system/medical_ocr&nlp_system
DATA_DIR exists = True
OCR_DIR exists = True
NER_GOLD_DIR = /content/medical_ocr&nlp_system/medical_ocr&nlp_system/data/ner_gold
OCR files count = 12


In [15]:
!pip install -U pip
!pip install -q "spacy[cuda11x]>=3.7,<4.0" tqdm
!python -m spacy validate


✔ Loaded compatibility table

================ Installed pipeline packages (spaCy v3.8.14) ================
ℹ spaCy installation: /usr/local/lib/python3.12/dist-packages/spacy

NAME             SPACY            VERSION                            
en_core_web_sm   >=3.8.0,<3.9.0   3.8.0   ✔



In [16]:
from pathlib import Path

annotated_path = NER_GOLD_DIR / 'annotated.jsonl'
bootstrap_path = NER_GOLD_DIR / 'bootstrap_weak.jsonl'

if annotated_path.exists():
    SOURCE_JSONL = annotated_path
    print('Using annotated.jsonl')
elif bootstrap_path.exists():
    SOURCE_JSONL = bootstrap_path
    print('Using bootstrap_weak.jsonl (a corriger manuellement conseillé)')
else:
    raise FileNotFoundError('Ni annotated.jsonl ni bootstrap_weak.jsonl trouvés.')

print('SOURCE_JSONL =', SOURCE_JSONL)

Using annotated.jsonl
SOURCE_JSONL = /content/medical_ocr&nlp_system/medical_ocr&nlp_system/data/ner_gold/annotated.jsonl


In [19]:
import json, random
from collections import Counter, defaultdict
from pathlib import Path
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans

# Fix for numpy.dtype size changed error and Cython compatibility
# Aligning with cuda11x as per previous successful installation
!pip uninstall -y numpy spacy thinc
!pip install -q "spacy[cuda11x]>=3.7,<4.0"

ALLOWED_LABELS = {
    "TEST_NAME", "VALUE", "UNIT", "REFERENCE_RANGE", "FLAG",
    "PATIENT_ID", "DATE_TIME", "DOCTOR", "SAMPLE_NO"
}

train_jsonl = NER_GOLD_DIR / 'train.jsonl'
dev_jsonl = NER_GOLD_DIR / 'dev.jsonl'
train_spacy = NER_GOLD_DIR / 'train.spacy'
dev_spacy = NER_GOLD_DIR / 'dev.spacy'

records = []
with SOURCE_JSONL.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        records.append(json.loads(line))

valid = []
dropped = 0
label_counter = Counter()
for r in records:
    text = r.get('text', '')
    ents = r.get('entities', [])
    if not text or not isinstance(ents, list):
        dropped += 1
        continue

    cleaned = []
    for e in ents:
        s, en, lb = e.get('start'), e.get('end'), e.get('label')
        if not isinstance(s, int) or not isinstance(en, int):
            continue
        if s < 0 or en > len(text) or en <= s:
            continue
        if lb not in ALLOWED_LABELS:
            continue
        if text[s:en].strip() == '':
            continue
        cleaned.append({'start': s, 'end': en, 'label': lb})

    cleaned = sorted(cleaned, key=lambda x: (x['start'], x['end']))
    non_overlap = []
    last_end = -1
    for e in cleaned:
        if e['start'] >= last_end:
            non_overlap.append(e)
            last_end = e['end']

    if not non_overlap:
        dropped += 1
        continue

    for e in non_overlap:
        label_counter[e['label']] += 1

    valid.append({
        'text': text,
        'entities': non_overlap,
        'source_file': r.get('source_file', 'unknown')
    })

# split par source_file pour éviter fuite train/dev
by_file = defaultdict(list)
for r in valid:
    by_file[r['source_file']].append(r)

files = list(by_file.keys())
random.seed(42)
random.shuffle(files)

if len(files) <= 1:
    train_records = valid
    dev_records = []
else:
    dev_count = max(1, round(0.2 * len(files)))
    dev_files = set(files[:dev_count])
    train_records = [r for f in files if f not in dev_files for r in by_file[f]]
    dev_records = [r for f in files if f in dev_files for r in by_file[f]]

with train_jsonl.open('w', encoding='utf-8') as f:
    for r in train_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

with dev_jsonl.open('w', encoding='utf-8') as f:
    for r in dev_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

nlp = spacy.blank('en')

def to_docbin(items, out_path):
    db = DocBin(store_user_data=True)
    dropped_spans = 0
    for r in items:
        doc = nlp.make_doc(r['text'])
        spans = []
        for e in r['entities']:
            sp = doc.char_span(e['start'], e['end'], label=e['label'], alignment_mode='contract')
            if sp is None:
                dropped_spans += 1
                continue
            spans.append(sp)
        doc.ents = filter_spans(spans)
        db.add(doc)
    db.to_disk(out_path)
    return dropped_spans

train_dropped = to_docbin(train_records, train_spacy)
dev_dropped = to_docbin(dev_records, dev_spacy)

print('raw records      =', len(records))
print('valid records    =', len(valid))
print('dropped records  =', dropped)
print('train records    =', len(train_records))
print('dev records      =', len(dev_records))
print('train drop spans =', train_dropped)
print('dev drop spans   =', dev_dropped)
print('label distribution:', dict(label_counter))

KeyError: '__reduce_cython__'